# M-PESA FRAUD DETECTION

In Kenya, the fintech industry in money transactions is primarily dominated by mobile money — **M-Pesa** to be exact.

Launched in 2007, M-Pesa tackled a systemic and infrastructural gap the African banking system couldn't close. While banking institutions adopted Western-developed infrastructure architectures, the majority of Kenyans were left with limited access to financial services — banks only had branches in major towns and cities. M-Pesa changed this entirely: all you needed was a SIM card. With agents operating in deep rural areas across the country, money could reach the most remote corners of Kenya within seconds, compared to money orders that could take over three days.

Today, Kenya's economy is built around M-Pesa. It is embedded in transactions as low as KSh 1 — across SMEs, small local shops, kiosks, mobile-to-bank transfers, and everyday essentials like buying airtime, paying for fare, or sending money to *mama mboga*.

> **Dataset Disclaimer:** This notebook uses a synthetic M-Pesa fraud dataset that mimics real transaction behaviour across different categorical variables. See the **Dataset** section for scope and limitations.

## PROBLEM STATEMENT

As M-Pesa came to dominate money transactions in Kenya, so did the volume of fraudulent activity targeting its users. Common attack vectors include **SIM swaps**, **social engineering**, and **fraudulent merchant transactions**. Fraudsters continuously adapt their tactics — deploying number masking, fake balance SMS traps, and agent impersonation to exploit both everyday users and M-Pesa agents.

### The Scale of the Problem

Safaricom's FY2024 Annual Report places the total value of M-Pesa transactions at **KSh 38.29 trillion annually**, representing **37.15 billion individual transactions** over a 12-month period — making M-Pesa the largest fintech platform on the African continent.

TransUnion's H2 2024 Consumer Pulse Study reports that the median loss per successful digital scam in Kenya is **KSh 108,132** — the highest median financial loss recorded among all African countries surveyed.

Applying a conservative network fraud rate of **0.006%** to Safaricom's reported transaction volume yields a systemic annual loss estimate of **KSh 2.30 billion (~$17.6 million USD)**.

### Mathematical Breakdown

| Metric | Value |
|---|---|
| Total Transaction Value | KSh 38,290,000,000,000 |
| Estimated Network Fraud Rate | 0.006% |
| Annual Systemic Loss | KSh 2,297,400,000 |

### Putting the Losses in Context

- **Daily drainage:** ~KSh 6.30 million stolen from the ecosystem every single day
- **Hourly theft:** ~KSh 262,260 lost to fraud networks every hour
- **Individual impact:** At the TransUnion median loss of KSh 108,132, this figure represents roughly **21,246 successful scam operations** hitting Kenyan users annually

### Why This Is a Conservative Floor

The KSh 2.3 billion estimate captures direct systemic loss — actual socioeconomic damage is likely significantly higher:

- **Underreporting:** A large proportion of P2P fraud victims do not report losses under KSh 5,000 to Safaricom or law enforcement, either out of embarrassment or a belief that recovery is unlikely
- **Secondary losses:** The estimate does not capture funds stolen from bank accounts linked to M-Pesa via mobile banking integrations
- **Collateral business damage:** For small merchants, fraud-related losses frequently lead to business closure, credit score damage, or legal costs that never appear in network transaction data

## GOAL

With fraudulent transactions costing the Kenyan economy billions annually, this project builds an **end-to-end machine learning pipeline** that classifies whether a transaction should be flagged as fraudulent — in real time, within the latency window of an M-Pesa transaction initiation.

Rather than logging fraud after the fact, this pipeline is designed to **intercept transactions before settlement**. The model sits between transaction initiation and transaction completion, returning one of three decisions:

| Decision | Condition |
|---|---|
| `ALLOW` | Transaction meets no fraud threshold — proceed normally |
| `CHALLENGE` | Transaction exhibits elevated risk — trigger OTP or user verification |
| `BLOCK` | Transaction meets fraud threshold — halt before settlement |

This positions the model not as a detection audit tool, but as a **real-time transaction decisioning engine** — one that protects customers before money leaves their account, and helps platforms like M-Pesa strengthen the trust that underpins their entire value proposition.

## DATASET

**Source:** Synthetic M-Pesa Fraud Dataset — [Kaggle (calebboen)](https://www.kaggle.com/datasets/calebboen/mpesa-transactions-fraud)  
**Size:** 120,000 transactions × 13 features  
**Target variable:** `is_fraud` (binary: 0 = legitimate, 1 = fraudulent)  
**Overall fraud rate:** ~2.93% (3,510 fraudulent transactions)

### Features

| Feature | Type | Description |
|---|---|---|
| `transaction_id` | string | Unique transaction identifier |
| `amount` | float | Transaction amount (KSh) |
| `sender_balance_before` | float | Sender wallet balance before transaction |
| `sender_balance_after` | float | Sender wallet balance after transaction |
| `receiver_balance_before` | float | Receiver wallet balance before transaction |
| `receiver_balance_after` | float | Receiver wallet balance after transaction |
| `transaction_type` | categorical | peer / till / paybill |
| `hour` | int | Hour of transaction (0–23) |
| `month_2026` | int | Month of transaction |
| `day_of_week` | categorical | Day name (Mon–Sun) |
| `device_type` | categorical | feature / smartphone |
| `region` | categorical | Nairobi / Mombasa / Kisumu / Nakuru / Eldoret |
| `is_fraud` | int | Target label (0 or 1) |

### Known Limitations

- **Uniform fraud distribution:** Fraud rates are nearly identical across all categorical variables (~2.9% per slice). Real-world fraud patterns exhibit strong variation — e.g., higher rates on feature phones (SIM swap vector) and overnight transactions. This is a known characteristic of synthetic data and is addressed in the EDA.
- **Inflated fraud rate:** The dataset's 2.93% fraud rate is approximately 490× higher than Safaricom's estimated network fraud rate of 0.006%. This is intentional for model training — sufficient fraud examples are required for the classifier to learn — but should not be interpreted as a realistic prevalence estimate.
- **No agent transaction type:** M-Pesa agent float abuse is a documented fraud vector not represented in this dataset.
- **No SIM swap flag:** SIM swap fraud — one of the most damaging attack types — is not explicitly labelled as a transaction subtype.

These limitations are documented in the model card and inform the scope of future work.

## DELIVERABLES

1. **EDA notebook** — exploratory analysis with business insights, fraud pattern visualisations, and feature distribution analysis
2. **Feature engineering** — balance drain ratio, cyclical time encoding, and derived behavioural features documented with rationale
3. **End-to-end ML pipeline** — data ingestion → preprocessing → feature engineering → model training → evaluation
4. **Class imbalance handling** — SMOTE oversampling and cost-sensitive learning, with comparison of impact on model performance
5. **Model comparison** — Logistic Regression, Random Forest, XGBoost, LightGBM evaluated on precision, recall, F1, and AUC-ROC
6. **FastAPI decisioning endpoint** — real-time transaction scoring returning fraud probability, risk tier, recommended action (`ALLOW/CHALLENGE/BLOCK`), and action rationale
7. **Model card** — performance metrics, feature importance, known biases, dataset limitations, and future expansion scope

In [ ]:
# Library Importations
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# Loading data to memory
data = pd.read_csv('../Data/mpesa_synthetic.csv')
data.head()

,transaction_id,amount,sender_balance_before,sender_balance_after,receiver_balance_before,receiver_balance_after,transaction_type,hour,month_2026,day_of_week,device_type,region,is_fraud
0,UA0IFD0TV,703.90,58586.32,57882.42,29932.92,30636.82,peer,14,1,Tue,smartphone,Nairobi,0
1,UJAHXTHV3,254.44,8088.00,7833.56,22962.44,23216.88,peer,18,10,Sat,feature,Eldoret,1
2,UEF8MDD4V,609.04,56675.00,56065.96,1029.22,1638.26,till,7,5,Thu,smartphone,Kisumu,0
3,UBT3W5UZB,5255.34,75090.36,69835.02,38.94,5294.28,paybill,11,2,Mon,smartphone,Nakuru,0
4,UGKWNNHJ7,7282.67,24408.96,17126.29,26237.82,33520.49,till,0,7,Sat,smartphone,Nairobi,0


In [ ]:
# Data overview
print(f"{'='*100}\nTotal Null Values per Column: {data.isna().sum()}")
print(f"{'='*100}\nData Shape: {data.shape}")
print('='*100)
print(f"{'='*100}\nData infomation including data types: \n{data.info()}")
print(f"{'='*100}\nStatistical Overview: \n{data.describe()}")
print(f"{'='*100}\nColumns:\n{data.columns}")

Total Null Values per Column: transaction_id             0
amount                     0
sender_balance_before      0
sender_balance_after       0
receiver_balance_before    0
receiver_balance_after     0
transaction_type           0
hour                       0
month_2026                 0
day_of_week                0
device_type                0
region                     0
is_fraud                   0
dtype: int64
Data Shape: (120000, 13)
<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   transaction_id           120000 non-null  str    
 1   amount                   120000 non-null  float64
 2   sender_balance_before    120000 non-null  float64
 3   sender_balance_after     120000 non-null  float64
 4   receiver_balance_before  120000 non-null  float64
 5   receiver_balance_after   120000 non-null  float64
 6   transactio

In [ ]:
print(data.info())

<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   transaction_id           120000 non-null  str    
 1   amount                   120000 non-null  float64
 2   sender_balance_before    120000 non-null  float64
 3   sender_balance_after     120000 non-null  float64
 4   receiver_balance_before  120000 non-null  float64
 5   receiver_balance_after   120000 non-null  float64
 6   transaction_type         120000 non-null  str    
 7   hour                     120000 non-null  int64  
 8   month_2026               120000 non-null  int64  
 9   day_of_week              120000 non-null  str    
 10  device_type              120000 non-null  str    
 11  region                   120000 non-null  str    
 12  is_fraud                 120000 non-null  int64  
dtypes: float64(5), int64(3), str(5)
memory usage: 11.9 MB
None
